
# Experiment 3.11: Product-Space Muon

## Literate counterpart to `run_experiment.py`

This notebook is the explanatory companion to
`experiments/Tier_1_Core_Mechanism_Tests/PRODUCT_SPACE_muon/run_experiment.py`.
It does **not** duplicate the training loop. Instead, it imports and runs the
script's `run_experiment()` entrypoint and analyzes the returned structured
results.

### Truthful scope
- **Toy setting:** deep linear regression.
- **Implemented measurements:** training loss and end-to-end product condition number.
- **Not measured here:** gradient propagation, feature learning, significance testing, or large-model behavior.

### Pair-level question
Does periodic product-space SVD rebalancing mainly help by controlling the
end-to-end product condition number, and if so, does that conditioning control
also translate into lower final loss than plain Muon?



## Benchmark definition used here

To fix the clearest truthfulness problem from the earlier version, the notebook
and script now use the same seeded **teacher-map** task:

\[
W_{\mathrm{target}} = U\,\mathrm{diag}(\sigma)\,V^\top,\qquad
Y_{\mathrm{target}} = W_{\mathrm{target}} X,
\]

with singular values ranging from `100` down to `1`. Because `X` is drawn from
the same per-seed RNG and `Y_target = W_target @ X`, the effective learned map
is the claimed teacher map itself, so the reported target condition number is
honest.


In [ ]:

import json
import os
import platform
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')


def find_repo_root(start: Path) -> Path:
    target = Path('experiments/Tier_1_Core_Mechanism_Tests/PRODUCT_SPACE_muon/run_experiment.py')
    for candidate in [start, *start.parents]:
        if (candidate / target).exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate repo root containing '
        'experiments/Tier_1_Core_Mechanism_Tests/PRODUCT_SPACE_muon/run_experiment.py. '
        'Run this notebook from inside the repository tree.'
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'PRODUCT_SPACE_muon'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)
print('Experiment directory:', EXPERIMENT_DIR)


In [ ]:

from experiments.Tier_1_Core_Mechanism_Tests.PRODUCT_SPACE_muon.run_experiment import (
    DEFAULT_CONFIG,
    METHOD_ORDER,
    compute_product_matrix,
    init_weights,
    layer_condition_numbers,
    make_teacher_problem,
    product_condition_number,
    rebalance_product,
    run_experiment,
)



## Reproducibility, runtime, and configuration

This cell logs the execution environment, shows the exact experiment config, and
runs the shared script entrypoint. By default it uses the script defaults. For a
faster smoke run, set the environment variable
`PRODUCT_SPACE_MUON_NOTEBOOK_OVERRIDES` to a JSON dictionary such as
`{"num_seeds": 1, "num_steps": 60, "lr_search_steps": 20}` before executing
this notebook.


In [ ]:

NOTEBOOK_RUN_CONFIG = DEFAULT_CONFIG.copy()
NOTEBOOK_RUN_OVERRIDES_JSON = os.environ.get('PRODUCT_SPACE_MUON_NOTEBOOK_OVERRIDES')
if NOTEBOOK_RUN_OVERRIDES_JSON:
    NOTEBOOK_RUN_CONFIG.update(json.loads(NOTEBOOK_RUN_OVERRIDES_JSON))

NOTEBOOK_VERBOSE = bool(NOTEBOOK_RUN_CONFIG.pop('verbose', False))

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('NumPy:', np.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Experiment script:', EXPERIMENT_DIR / 'run_experiment.py')
print('Notebook run config:')
print(json.dumps(NOTEBOOK_RUN_CONFIG, indent=2))

results = run_experiment(verbose=NOTEBOOK_VERBOSE, **NOTEBOOK_RUN_CONFIG)

print(f"Measured experiment runtime: {results['runtime_seconds']:.2f} seconds")
print('Executed seeds:', results['seeds'])
print('Rebalancing steps:', results['config']['rebalancing_steps'])


In [ ]:

METHOD_LABELS = {method: results['methods'][method]['label'] for method in METHOD_ORDER}


def curves(method, key):
    return np.asarray(results['methods'][method][key], dtype=float)


def sem(array, axis=0):
    array = np.asarray(array, dtype=float)
    if array.shape[axis] <= 1:
        return np.zeros_like(np.mean(array, axis=axis))
    return np.std(array, axis=axis, ddof=1) / np.sqrt(array.shape[axis])


def sci(x):
    return f'{x:.3e}'


steps = np.arange(results['config']['num_steps'] + 1)
rebalancing_steps = results['config']['rebalancing_steps']
colors = {'sgd': 'tab:blue', 'muon': 'tab:orange', 'muon_rebalance': 'tab:green'}



## Mechanism sanity check: product preservation and layerwise equalization

The core claim of product-space rebalancing is purely algebraic: rebalance the
singular values of the full product without changing the full product itself.
The following cell reconstructs the **exact first-seed initialization path** used
by the experiment and checks that behavior on one sampled stack.


In [ ]:

cfg = results['config']
sample_seed = results['seeds'][0]
rng = np.random.RandomState(sample_seed)
_ = make_teacher_problem(
    width=cfg['width'],
    batch_size=cfg['batch_size'],
    rng=rng,
    target_condition_number=cfg['target_condition_number'],
    input_scale=cfg['input_scale'],
)
sample_weights = init_weights(cfg['num_layers'], cfg['width'], rng)

prod_before = compute_product_matrix(sample_weights)
kappa_prod_before = product_condition_number(sample_weights)
layer_kappas_before = np.asarray(layer_condition_numbers(sample_weights), dtype=float)

rebalanced_weights = rebalance_product(sample_weights)
prod_after = compute_product_matrix(rebalanced_weights)
kappa_prod_after = product_condition_number(rebalanced_weights)
layer_kappas_after = np.asarray(layer_condition_numbers(rebalanced_weights), dtype=float)

rel_product_error = np.linalg.norm(prod_after - prod_before) / np.linalg.norm(prod_before)
expected_layer_kappa = kappa_prod_before ** (1.0 / cfg['num_layers'])
equalization_ok = np.allclose(layer_kappas_after, expected_layer_kappa, rtol=1e-5, atol=1e-8)

print('Mechanism sanity check on first-seed initialization')
print('  Relative product error after rebalance:', f'{rel_product_error:.3e}')
print('  Product kappa before rebalance:', f'{kappa_prod_before:.6e}')
print('  Product kappa after rebalance: ', f'{kappa_prod_after:.6e}')
print('  Mean layer kappa before rebalance:', f'{layer_kappas_before.mean():.6e}')
print('  Mean layer kappa after rebalance: ', f'{layer_kappas_after.mean():.6e}')
print('  Expected post-rebalance layer kappa:', f'{expected_layer_kappa:.6e}')
print('  Product preservation check:', 'PASS' if rel_product_error < 1e-10 else 'CHECK')
print('  Equalization check:', 'PASS' if equalization_ok else 'CHECK')
print()
print('Layer kappas before rebalance:')
print(np.array2string(layer_kappas_before, precision=3, suppress_small=False))
print()
print('Layer kappas after rebalance:')
print(np.array2string(layer_kappas_after, precision=3, suppress_small=False))



## Per-seed setup and selected learning rates

This table makes the benchmark definition explicit for each seed. In particular,
it shows that the nominal teacher-map condition number and the effective learned
map condition number match up to numerical precision.


In [ ]:

print('Seed | init product kappa | target kappa | effective map kappa | LR(SGD) | LR(Muon) | LR(Muon+RB)')
print('-' * 110)
for i, seed in enumerate(results['seeds']):
    print(
        f"{seed:>4d} | "
        f"{results['initial_product_kappas'][i]:>18.3e} | "
        f"{results['target_condition_numbers'][i]:>12.3e} | "
        f"{results['effective_map_condition_numbers'][i]:>18.3e} | "
        f"{results['selected_lrs']['sgd'][i]:>7.4f} | "
        f"{results['selected_lrs']['muon'][i]:>8.4f} | "
        f"{results['selected_lrs']['muon_rebalance'][i]:>11.4f}"
    )



## Aggregate final metrics across seeds

The benchmark's direct outputs are the final losses and final end-to-end product
condition numbers. Means, standard deviations, and standard errors are reported
below.


In [ ]:

print('Method                         | Mean final loss | Std loss      | SEM loss      | Mean final kappa | Std kappa     | SEM kappa')
print('-' * 130)
for method in METHOD_ORDER:
    stats = results['aggregate'][method]
    print(
        f"{METHOD_LABELS[method]:<30} | "
        f"{stats['mean_final_loss']:>15.6e} | "
        f"{stats['std_final_loss']:>13.6e} | "
        f"{stats['sem_final_loss']:>13.6e} | "
        f"{stats['mean_final_kappa']:>16.6e} | "
        f"{stats['std_final_kappa']:>13.6e} | "
        f"{stats['sem_final_kappa']:>13.6e}"
    )



## Mean trajectory of end-to-end conditioning

The central object of interest is the product condition number
\(\kappa_{\mathrm{prod}}(W_L\cdots W_1)\). The plot below shows the mean
\(\log_{10}\kappa_{\mathrm{prod}}\) across seeds together with a SEM band.
Vertical dashed lines mark rebalance steps.


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

for method in METHOD_ORDER:
    kappa_curves = np.clip(curves(method, 'kappa_curves'), 1e-30, None)
    log_kappa = np.log10(kappa_curves)
    mean_curve = log_kappa.mean(axis=0)
    sem_curve = sem(log_kappa, axis=0)
    ax.plot(steps, mean_curve, label=METHOD_LABELS[method], color=colors[method], linewidth=2)
    ax.fill_between(steps, mean_curve - sem_curve, mean_curve + sem_curve, color=colors[method], alpha=0.2)

if rebalancing_steps:
    ax.axvline(rebalancing_steps[0], color='gray', linestyle='--', alpha=0.6, label='rebalance step')
    for step in rebalancing_steps[1:]:
        ax.axvline(step, color='gray', linestyle='--', alpha=0.25)

ax.set_xlabel('Training step')
ax.set_ylabel(r'Mean $\log_{10}\,\kappa_{\mathrm{prod}}$ $\pm$ SEM')
ax.set_title('End-to-end product conditioning across training')
ax.legend()
plt.tight_layout()
plt.show()



## Mean trajectory of training loss

Loss is plotted on a log scale because the SGD baseline can remain orders of
magnitude worse than the Muon variants. This plot is the notebook's main check
for whether conditioning improvements actually translate into optimization
improvements.


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))

for method in METHOD_ORDER:
    loss_curves = np.clip(curves(method, 'loss_curves'), 1e-12, None)
    mean_curve = loss_curves.mean(axis=0)
    sem_curve = sem(loss_curves, axis=0)
    lower = np.clip(mean_curve - sem_curve, 1e-12, None)
    upper = np.clip(mean_curve + sem_curve, 1e-12, None)
    ax.plot(steps, mean_curve, label=METHOD_LABELS[method], color=colors[method], linewidth=2)
    ax.fill_between(steps, lower, upper, color=colors[method], alpha=0.2)

if rebalancing_steps:
    ax.axvline(rebalancing_steps[0], color='gray', linestyle='--', alpha=0.6, label='rebalance step')
    for step in rebalancing_steps[1:]:
        ax.axvline(step, color='gray', linestyle='--', alpha=0.25)

ax.set_yscale('log')
ax.set_xlabel('Training step')
ax.set_ylabel('Mean loss ± SEM (log scale)')
ax.set_title('Training loss across training')
ax.legend()
plt.tight_layout()
plt.show()



## Paired per-seed comparison: Muon vs Muon+Rebalance

Aggregate means can hide whether one method consistently wins seed-by-seed. The
next figure and table therefore compare the final loss and final product
condition number **pairwise** for Muon and Muon+Rebalance.


In [ ]:

muon_final_loss = np.asarray(results['methods']['muon']['final_losses'], dtype=float)
rb_final_loss = np.asarray(results['methods']['muon_rebalance']['final_losses'], dtype=float)
muon_final_kappa = np.asarray(results['methods']['muon']['final_kappas'], dtype=float)
rb_final_kappa = np.asarray(results['methods']['muon_rebalance']['final_kappas'], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.array([0, 1])
labels = ['Muon', 'Muon+RB']

for i, seed in enumerate(results['seeds']):
    axes[0].plot(x, [muon_final_loss[i], rb_final_loss[i]], marker='o', linewidth=1.5, alpha=0.8)
    axes[1].plot(x, [muon_final_kappa[i], rb_final_kappa[i]], marker='o', linewidth=1.5, alpha=0.8)
    axes[0].text(1.02, rb_final_loss[i], str(seed), fontsize=8, va='center')
    axes[1].text(1.02, rb_final_kappa[i], str(seed), fontsize=8, va='center')

axes[0].set_xticks(x, labels)
axes[1].set_xticks(x, labels)
axes[0].set_yscale('log')
axes[1].set_yscale('log')
axes[0].set_title('Final loss by seed')
axes[1].set_title('Final product kappa by seed')
axes[0].set_ylabel('Value (log scale)')
axes[1].set_ylabel('Value (log scale)')
plt.tight_layout()
plt.show()

loss_delta = rb_final_loss - muon_final_loss
log10_kappa_ratio = np.log10(rb_final_kappa / muon_final_kappa)

print('Seed | Muon loss | Muon+RB loss | RB-Muon loss | Muon kappa | Muon+RB kappa | log10(RB/Muon kappa)')
print('-' * 120)
for i, seed in enumerate(results['seeds']):
    print(
        f"{seed:>4d} | "
        f"{muon_final_loss[i]:>9.3e} | "
        f"{rb_final_loss[i]:>13.3e} | "
        f"{loss_delta[i]:>12.3e} | "
        f"{muon_final_kappa[i]:>10.3e} | "
        f"{rb_final_kappa[i]:>14.3e} | "
        f"{log10_kappa_ratio[i]:>20.3f}"
    )

print()
print('Mean paired deltas:')
print('  mean(RB - Muon final loss):', sci(loss_delta.mean()))
print('  mean log10(RB / Muon final kappa):', f'{log10_kappa_ratio.mean():.3f}')



## Crossover analysis and simple benchmark tests

The script's retained crossover diagnostic asks when a method's product kappa
first rises above SGD's and then stays above for more than 80% of the remaining
trajectory. The table below reports those crossover steps together with the four
simple tests tracked by the script.


In [ ]:

print('Seed | Muon crossover step | Muon+RB crossover step')
print('-' * 60)
for row in results['crossover']['per_seed']:
    print(
        f"{row['seed']:>4d} | "
        f"{str(row['muon_crossover_step']):>19} | "
        f"{str(row['muon_rebalance_crossover_step']):>24}"
    )

print()
print('No-crossover seed counts:')
print(
    f"  Muon = {results['crossover']['n_no_cross_muon']}/{results['config']['num_seeds']} | "
    f"Muon+RB = {results['crossover']['n_no_cross_muon_rebalance']}/{results['config']['num_seeds']}"
)

print()
print('Simple tests tracked by the script:')
for key in ('T1', 'T2', 'T3', 'T4'):
    test = results['tests'][key]
    status = 'PASS' if test['pass'] else 'FAIL'
    print(f"  {key}: {status} -- {test['description']}")
    print(f"      value = {test['value']} | reference = {test['reference']}")



## Calibrated conclusion

The conclusion below deliberately separates **conditioning control** from
**actual loss improvement**. This is the key rigor fix relative to the earlier
version, which could overclaim convergence improvement even when the loss metric
did not support it.


In [ ]:

verdict = results['verdict']

print('Headline:', verdict['headline'])
print()
print('Conditioning statement:')
print(verdict['conditioning_statement'])
print()
print('Loss statement relative to plain Muon:')
print(verdict['loss_statement'])
print()
print('Loss statement relative to SGD:')
print(verdict['baseline_statement'])
print()
print(f"Tests passed: {verdict['num_tests_passed']}/{verdict['num_tests']}")

if verdict['conditioning_supported'] and not verdict['loss_improvement_vs_muon_supported']:
    print()
    print('Bottom line: the current evidence supports better conditioning control, but not lower final loss than plain Muon.')
elif verdict['conditioning_supported'] and verdict['loss_improvement_vs_muon_supported']:
    print()
    print('Bottom line: the current evidence supports both conditioning control and lower final loss than plain Muon.')
else:
    print()
    print('Bottom line: this run does not establish a clear advantage for product-space rebalancing over plain Muon.')
